# 0 — Configure the sweeps (models, datasets, parameters)

**The one place to change *what* gets evaluated.** Edit the cells below, **Run All**
(takes a second, since it only validates and writes a small JSON file), then run the
sweep notebook(s). No image rebuild, no other edits.

| You edit here | Read by |
|---|---|
| `ADAPT` | `1_run_sweep_adapt` (CPU, CIFAR CNNs) |
| `TRANSAXX` | `1_run_sweep_transaxx` (GPU, CIFAR CNNs, the cross-validation of AdaPT) |
| `TRANSAXX_VIT` | `1_run_sweep_transaxx_vit` (GPU, ImageNet ViTs) |
| `LOCAL_MODELS` | `1_run_sweep_adapt` (your own model files, see below) |

Everything lands in `sweep_config.json` next to the notebooks. Each sweep notebook reads
it at the top and falls back to its built-in defaults if the file is missing.

**Not configured here, on purpose:**
- *Which multipliers* — define those in `0_build_multipliers.ipynb` (the sweeps pick up
  its `multipliers.json` manifest automatically).
- *`FRESH`* — the toggle that **deletes** existing results stays inside each sweep
  notebook, so a shared config file can never wipe results as a side effect.

In [1]:
# ============ AdaPT sweep (CPU) — CIFAR CNNs ============
# models must be pretrained checkpoints from chenyaofo/pytorch-cifar-models
# (the validation cell below knows the legal names). Datasets: cifar10, cifar100.
ADAPT = {
    "datasets":       ["cifar10", "cifar100"],
    "models":         ["resnet20", "resnet56", "vgg11_bn"],
    "retrain_epochs": 3,     # 0 = eval-only
    "batch_size":     128,
    "calib_batches":  2,     # calibration batches for the int8 quantizer
}

In [2]:
# ============ TransAxx sweep (GPU) — CIFAR CNNs ============
# Same legal models/datasets as AdaPT (it is the GPU cross-validation of the same matrix).
TRANSAXX = {
    "datasets":       ["cifar10", "cifar100"],
    "models":         ["resnet20", "resnet56", "vgg11_bn"],
    "retrain_epochs": 3,
    "batch_size":     128,
    "calib_batches":  2,
}

In [3]:
# ============ TransAxx ViT sweep (GPU) — ImageNet ============
# models can be ANY timm name pretrained on ImageNet-1k (https://huggingface.co/timm).
# e.g. add "deit_base_patch16_224" and it just works: the sweep resolves each
# model's own preprocessing (mean/std/crop/interpolation) from timm automatically.
#   dataset: "imagenet" -> the real ILSVRC2012 val set (stage it with the ViT
#            notebook's "Prepare real ImageNet" cell)
TRANSAXX_VIT = {
    "models":         ["vit_small_patch16_224", "deit_small_patch16_224",
                       "vit_base_patch16_224"],
    "dataset":        "imagenet",
    "retrain_epochs": 0,     # eval-only; >0 would need the full ImageNet train set
    "batch_size":     128,
    "calib_batches":  8,     # ViTs need more calibration than CNNs
}

## Local models (your own `.pth` / externally provided)

A `.pth` holds only **weights**, not the architecture, so a local model is **two files**
in `docker/models/` (mounted at `../models` from this notebook's point of view):

```
docker/models/<name>.py     <- defines build_model() -> eval-ready model (copy _template.py)
docker/models/<name>.pth    <- the weights; <name>.py loads them itself
```

Then register it below and Run All. The AdaPT sweep picks it up automatically and runs it
through the full multiplier matrix **on its declared dataset only** (a model trained on
CIFAR-10 has no meaningful CIFAR-100 accuracy).

The `.py` contract (full example in `../models/_template.py`):
- `build_model()` returns the model with weights loaded, in eval mode;
- `forward` returns **one** logits tensor `[batch, n_classes]` (early-exit models must
  return the exit to score);
- loadable on CPU PyTorch 1.13 (AdaPT's environment). `framework` must be `"adapt"`
  for now, since that's the only sweep with the local-model hook wired in.

A custom **ViT** does not go through this hook. For that route see
`../models/_template_vit.py` and the *Extending the pipeline* cell below.

In [4]:
LOCAL_MODELS = {
    # "my_model": {"framework": "adapt", "dataset": "cifar10"},
}

## Extending the pipeline (datasets & off-contract models)

The config picks what runs by name. Two things need a small, documented code edit instead
of a config entry, and that is on purpose: the config is a whitelist, so a typo dies in the
validator below in seconds rather than an hour into a sweep, and the data provenance stays
explicit.

- **A new dataset.** Add one entry to the `DATASETS` registry at the top of
  `1_run_sweep_adapt` (norm stats, class count, and a `build(root, transform)` that returns
  `(train, test)`), then allow its name in the validator below. Anything torchvision can
  build, or your own `ImageFolder` under `docker/datasets/<name>_data/`, drops in the same
  way. There is a commented `myset` example in that cell. The data lives under
  `docker/datasets/` (bind-mounted), so it persists and can be staged offline.

- **A model that breaks the generic contract.** The AdaPT sweep expects one logits tensor
  `[batch, n_classes]` and standard per-channel-normalised CIFAR. A model with multiple
  outputs (early-exit) or its own preprocessing gets its own notebook, copied from
  `_template_sweep` — the standard AdaPT sweep with the model, the loaders and the scoring
  left as three marked `TODO`s. For a ViT the working example is `1_run_sweep_transaxx_vit`,
  which produces real results and so doubles as a template.

- **Your own ViT.** The `LOCAL_MODELS` hook is only wired into the AdaPT sweep, so it will
  not carry a custom ViT. A stock timm ImageNet-1k model is already a config add (drop the
  name into `TRANSAXX_VIT` above). A ViT that timm cannot build, or one that needs its own
  preprocessing, gets its own notebook copied from `1_run_sweep_transaxx_vit`. The contract
  here is wider than the AdaPT one: on top of `build_model()` you also hand over the eval
  transform and the layers that get approximated, since the timm path normally works those
  out from the model itself. `../models/_template_vit.py` writes that contract out.

**Already config-only (no code):** the multipliers (from `0_build_multipliers`), the sweep
params, the supported models and datasets above, and a custom model that *does* fit the
contract, via `LOCAL_MODELS` + `../models/_template.py`.

## Validate & write

Checks every entry against what the sweeps can actually run, then writes
`sweep_config.json`. A typo fails **here, in seconds**, instead of an hour into a sweep.

In [5]:
from pathlib import Path

# pretrained checkpoints chenyaofo/pytorch-cifar-models ships for BOTH cifar10 and
# cifar100 (mirrors its README, the hub id is "<dataset>_<model>")
CHENYAOFO_MODELS = {
    "resnet20", "resnet32", "resnet44", "resnet56",
    "vgg11_bn", "vgg13_bn", "vgg16_bn", "vgg19_bn",
    "mobilenetv2_x0_5", "mobilenetv2_x0_75", "mobilenetv2_x1_0", "mobilenetv2_x1_4",
    "shufflenetv2_x0_5", "shufflenetv2_x1_0", "shufflenetv2_x1_5", "shufflenetv2_x2_0",
    "repvgg_a0", "repvgg_a1", "repvgg_a2",
}
CIFAR_DATASETS = {"cifar10", "cifar100"}
VIT_DATASETS = {"imagenet"}
MODELS_DIR = Path("../models")   # docker/models on the host, /work/models in-container

problems = []

def check_common(name, cfg):
    for key in ("retrain_epochs", "batch_size", "calib_batches"):
        if not (isinstance(cfg.get(key), int) and cfg[key] >= 0):
            problems.append(f"{name}.{key} must be a non-negative integer, got {cfg.get(key)!r}")

for name, cfg in (("ADAPT", ADAPT), ("TRANSAXX", TRANSAXX)):
    check_common(name, cfg)
    for ds in cfg["datasets"]:
        if ds not in CIFAR_DATASETS:
            problems.append(f"{name}: dataset '{ds}' not supported (only {sorted(CIFAR_DATASETS)} "
                            f"have pretrained CIFAR checkpoints + loaders in this sweep)")
    for m in cfg["models"]:
        if m not in CHENYAOFO_MODELS:
            problems.append(f"{name}: model '{m}' is not a chenyaofo/pytorch-cifar-models "
                            f"checkpoint — pick from: {', '.join(sorted(CHENYAOFO_MODELS))}")

check_common("TRANSAXX_VIT", TRANSAXX_VIT)
if TRANSAXX_VIT.get("dataset") not in VIT_DATASETS:
    problems.append(f"TRANSAXX_VIT.dataset must be one of {sorted(VIT_DATASETS)}, "
                    f"got {TRANSAXX_VIT.get('dataset')!r} (real ILSVRC2012, staged by the "
                    f"ViT notebook's Prepare real ImageNet cell)")
for m in TRANSAXX_VIT["models"]:
    if not (isinstance(m, str) and m):   # can't resolve timm names without timm installed;
        problems.append(f"TRANSAXX_VIT: bad model entry {m!r}")
print("note: ViT names can't be fully checked here (timm not installed in analysis) — "
      "a wrong name fails fast in the sweep's first cell, before any evaluation.")

for name, spec in LOCAL_MODELS.items():
    if spec.get("framework") != "adapt":
        problems.append(f"LOCAL_MODELS['{name}']: framework must be 'adapt' (the only sweep "
                        f"with the local-model hook), got {spec.get('framework')!r}")
    if spec.get("dataset") not in CIFAR_DATASETS:
        problems.append(f"LOCAL_MODELS['{name}']: dataset must be one of {sorted(CIFAR_DATASETS)} "
                        f"(the AdaPT sweep's loaders), got {spec.get('dataset')!r}")
    py = MODELS_DIR / f"{name}.py"
    if not py.exists():
        problems.append(f"LOCAL_MODELS['{name}']: {py} not found — copy _template.py to "
                        f"{name}.py and put the weights next to it (see ../models/README.md)")
    elif "def build_model" not in py.read_text():
        problems.append(f"LOCAL_MODELS['{name}']: {py} doesn't define build_model() — "
                        f"see ../models/_template.py for the contract")

if problems:
    raise ValueError("config problems found:\n  - " + "\n  - ".join(problems))
print("all checks passed ✔")

note: ViT names can't be fully checked here (timm not installed in analysis) — a wrong name fails fast in the sweep's first cell, before any evaluation.
all checks passed ✔


In [6]:
import json

CONFIG_PATH = Path("sweep_config.json")   # next to the notebooks -> read by every sweep

config = {
    "_": "Written by 0_configure_sweeps.ipynb — edit THERE (this file is overwritten).",
    "adapt": ADAPT,
    "transaxx": TRANSAXX,
    "transaxx_vit": TRANSAXX_VIT,
    "local_models": LOCAL_MODELS,
}
CONFIG_PATH.write_text(json.dumps(config, indent=2) + "\n")
print(f"wrote {CONFIG_PATH.resolve()}\n")

_adapt_local = [n for n, s in LOCAL_MODELS.items() if s["framework"] == "adapt"]
print("resulting sweep matrices (x multipliers from 0_build_multipliers' manifest):")
print(f"  adapt        {ADAPT['datasets']} x {ADAPT['models']}"
      + (f" + local {_adapt_local}" if _adapt_local else "")
      + f"  (retrain_epochs={ADAPT['retrain_epochs']})")
print(f"  transaxx     {TRANSAXX['datasets']} x {TRANSAXX['models']}  "
      f"(retrain_epochs={TRANSAXX['retrain_epochs']})")
print(f"  transaxx_vit {TRANSAXX_VIT['dataset']} x {TRANSAXX_VIT['models']}  "
      f"(retrain_epochs={TRANSAXX_VIT['retrain_epochs']}, + fp32 baseline per model)")
print("\nnow run the sweep notebook(s) — they pick this file up automatically.")

wrote /work/notebooks/sweep_config.json

resulting sweep matrices (x multipliers from 0_build_multipliers' manifest):
  adapt        ['cifar10', 'cifar100'] x ['resnet20', 'resnet56', 'vgg11_bn']  (retrain_epochs=3)
  transaxx     ['cifar10', 'cifar100'] x ['resnet20', 'resnet56', 'vgg11_bn']  (retrain_epochs=3)
  transaxx_vit imagenet x ['vit_small_patch16_224', 'deit_small_patch16_224', 'vit_base_patch16_224']  (retrain_epochs=0, + fp32 baseline per model)

now run the sweep notebook(s) — they pick this file up automatically.
